# APS360 — SYKE IFCB exploratory data analysis
Emily (Jiarui) Hu · HAB early-warning project

This notebook combines the two EDA scripts used for the proposal:

| Part | What it does | Data needed |
|---|---|---|
| **A. Info-sheet EDA** | Summary numbers and 4 figures for the 2016–2019 training data and the 2021 test data | None (counts are typed in from the SYKE info sheet and Kraft et al. 2022, Table 1) |
| **B. 2021 folder counts** | Counts `.png` files in each class folder of the extracted 2021 dataset, then plots the class distribution | The extracted `phytoplankton_Utö_2021_labeled` folder (optional; falls back to the Table 1 counts) |

Runs on **Google Colab** or **locally** (Jupyter / VS Code on Windows). Edit the paths in the next cell only.

## 0. Setup and paths

In [ ]:
import os, sys, csv, time, statistics
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# ---- EDIT THESE PATHS ---------------------------------------------------------------
if IN_COLAB:
    # The extracted 2021 folder in your Google Drive (Part B). Wrong/missing path -> Table 1 counts are used.
    DATA_2021 = Path("/content/drive/MyDrive/APS 360 Project/SKYE_2021/phytoplankton_Utö_2021_labeled")
    # Figures and CSVs. /content is temporary; use a Drive path to keep them.
    OUT = Path("/content/eda_figures")
else:
    DATA_2021 = Path(r"C:\Users\withh\OneDrive\桌面\APS 360 Project\SKYE_2021\phytoplankton_Utö_2021_labeled")
    OUT = Path.cwd() / "eda_figures"
# -------------------------------------------------------------------------------------
OUT.mkdir(parents=True, exist_ok=True)
print("Colab" if IN_COLAB else "Local", "| 2021 folder found:", DATA_2021.is_dir(), "| output:", OUT)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
%matplotlib inline

# colours (validated reference palette, light mode)
C_HAB, C_NON, C_DROP, C_UNC = "#eb6834", "#2a78d6", "#b8b7b0", "#8a8983"
INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e6e5e0", "#ffffff"
plt.rcParams.update({"font.size": 10, "axes.edgecolor": INK2, "axes.labelcolor": INK,
                     "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.dpi": 200})
thousands = FuncFormatter(lambda x, _: f"{x:,.0f}")

def save_show(fig, name):
    fig.tight_layout()
    fig.savefig(OUT / name)
    plt.show()

## 1. Class table (shared by Parts A and B)
Counts per SYKE class: **2016–2019** from the `SYKE-plankton_IFCB_2022` info sheet; **2021** from Kraft et al. (2022), Table 1
(the 2021 info sheet lists class names only). Classes with fewer than 100 images in 2016–2019, and Beads, are dropped, leaving 38 SYKE classes.

In [ ]:
# (class name, taxonomic group, images 2016-2019, images 2021)
DATA = [
    ("Dolichospermum / Anabaenopsis",        "Cyanophyceae", 12280,   790),
    ("Aphanizomenon flosaquae",              "Cyanophyceae",  6989,  1849),
    ("Oscillatoriales",                      "Cyanophyceae",  4440,  3893),
    ("Snowella / Woronichinia",              "Cyanophyceae",  2950,    42),
    ("Dolichospermum / Anabaenopsis coiled", "Cyanophyceae",  2504,    70),
    ("Chroococcus sp.",                      "Cyanophyceae",   827,     2),
    ("Nodularia spumigena",                  "Cyanophyceae",   169,    62),
    ("Chroococcales",                        "Cyanophyceae",   142,   793),
    ("Merismopedia sp.",                     "Cyanophyceae",    98,     2),
    ("Aphanothece paralleliformis",          "Cyanophyceae",    29,     9),
    ("Cryptophyceae / Teleaulax sp.",        "Cryptophyceae", 6830, 16952),
    ("Cryptomonadales",                      "Cryptophyceae",  713,   525),
    ("Eutreptiella sp.",                     "Euglenophyceae", 2247, 1678),
    ("Euglenophyceae",                       "Euglenophyceae",  102,   18),
    ("Heterocapsa triquetra",                "Dinophyceae",   3276,  2267),
    ("Dinophyceae",                          "Dinophyceae",   1433,  1562),
    ("Peridiniella catenata single",         "Dinophyceae",    899,   222),
    ("Heterocapsa rotundata",                "Dinophyceae",    614,  2609),
    ("Prorocentrum cordatum",                "Dinophyceae",    276,     0),
    ("Dinophysis acuminata",                 "Dinophyceae",    217,    17),
    ("Peridiniella catenata chain",          "Dinophyceae",    193,    89),
    ("Gymnodinium like cells",               "Dinophyceae",    158,   102),
    ("Gymnodiniales",                        "Dinophyceae",     69,    38),
    ("Gonyaulax verior",                     "Dinophyceae",     22,     1),
    ("Amylax triacantha",                    "Dinophyceae",     19,     3),
    ("Skeletonema marinoi",                  "Diatoms",       4128,  7402),
    ("Thalassiosira levanderi",              "Diatoms",       2537,  2008),
    ("Chaetoceros sp. chain",                "Diatoms",       1382,   693),
    ("Pennales thin",                        "Diatoms",        781,   334),
    ("Centrales",                            "Diatoms",        480,    92),
    ("Chaetoceros sp. single",               "Diatoms",        213,   571),
    ("Pennales thick",                       "Diatoms",        210,  1088),
    ("Pauliella taeniata",                   "Diatoms",        119,    56),
    ("Cyclotella choctawhatcheeana",         "Diatoms",        102,   199),
    ("Licmophora sp.",                       "Diatoms",         74,    78),
    ("Nitzschia paleacea",                   "Diatoms",         65,     4),
    ("Ceratoneis closterium",                "Diatoms",         45,    75),
    ("Melosira arctica",                     "Diatoms",         43,    58),
    ("Uroglenopsis sp.",                     "Chrysophyceae",  516,   134),
    ("Pseudopedinella sp.",                  "Chrysophyceae",  379,   579),
    ("Pyramimonas sp.",                      "Chlorophyta",   1224,  8422),
    ("Oocystis sp.",                         "Chlorophyta",    842,   161),
    ("Monoraphidium contortum",              "Chlorophyta",    327,   439),
    ("Cymbomonas tetramitiformis",           "Chlorophyta",    199,     4),
    ("Chlorococcales",                       "Chlorophyta",     95,    45),
    ("Mesodinium rubrum",                    "Ciliates",      1132,   560),
    ("Ciliata",                              "Ciliates",       243,   288),
    ("Katablepharis remigera",               "Other",           54,     4),
    ("Heterocyte",                           "Additional",     263,   318),
    ("Beads",                                "Additional",     125,     0),
]
UNCLASSIFIABLE_2021 = 94028
HAB = {"Dolichospermum / Anabaenopsis", "Dolichospermum / Anabaenopsis coiled",
       "Aphanizomenon flosaquae", "Oscillatoriales", "Snowella / Woronichinia",
       "Nodularia spumigena", "Dinophysis acuminata", "Prorocentrum cordatum"}
MIN_IMAGES = 100          # classes below this (in 2016-2019) are dropped
EXCLUDE = {"Beads"}       # not plankton

def status(name, n_train):
    if name in EXCLUDE or n_train < MIN_IMAGES:
        return "dropped"
    return "HAB" if name in HAB else "non-HAB"

COLOURS = {"HAB": C_HAB, "non-HAB": C_NON, "dropped": C_DROP, "unclassifiable": C_UNC}
def colour(name, n_train):
    return COLOURS[status(name, n_train)]

rows = sorted(DATA, key=lambda r: r[2], reverse=True)
total_train = sum(r[2] for r in rows)
total_test = sum(r[3] for r in rows)
print(f"{len(rows)} classes | 2016-2019: {total_train:,} images | 2021: {total_test:,} classified")

## Part A. Info-sheet EDA
### A1. Summary numbers

In [ ]:
kept = [r for r in rows if status(r[0], r[2]) != "dropped"]
hab = [r for r in rows if r[0] in HAB]
counts = [r[2] for r in rows]
summary = [
    f"2016-2019 set: {total_train:,} images, {len(rows)} classes",
    f"Largest / smallest class: {rows[0][0]} {rows[0][2]:,} / {rows[-1][0]} {rows[-1][2]:,} "
    f"(imbalance {rows[0][2] / rows[-1][2]:.0f}:1)",
    f"Median class size: {statistics.median(counts):,.0f}; classes < 200 images: {sum(c < 200 for c in counts)}",
    f"Top 5 classes hold {100 * sum(counts[:5]) / total_train:.1f}% of all images",
    f"Kept (>= {MIN_IMAGES} images, no beads): {len(kept)} classes, {sum(r[2] for r in kept):,} images; "
    f"dropped {len(rows) - len(kept)} classes, {total_train - sum(r[2] for r in kept):,} images",
    f"HAB classes: {len(hab)}, {sum(r[2] for r in hab):,} images in 2016-2019 "
    f"({100 * sum(r[2] for r in hab) / total_train:.1f}%); {sum(r[3] for r in hab):,} in 2021 "
    f"({100 * sum(r[3] for r in hab) / total_test:.1f}% of classified)",
    f"2021: {total_test:,} classified + {UNCLASSIFIABLE_2021:,} unclassifiable images "
    f"({100 * UNCLASSIFIABLE_2021 / (total_test + UNCLASSIFIABLE_2021):.0f}% unclassifiable)",
    "Kept classes with < 10 test images in 2021: " + ", ".join(f"{r[0]} ({r[3]})" for r in kept if r[3] < 10),
]
(OUT / "summary.txt").write_text("\n".join(summary), encoding="utf-8")
print("\n".join(summary))

### A2. Class Distribution: SYKE 2016–2019 (Appendix Figure A1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 11))
names = [r[0] for r in rows][::-1]
vals = [r[2] for r in rows][::-1]
ax.barh(names, vals, color=[colour(r[0], r[2]) for r in rows][::-1], height=0.72)
ax.set_xscale("log")
ax.axvline(MIN_IMAGES, color=INK2, lw=1, ls="--")
ax.text(MIN_IMAGES * 1.06, -1.6, f"cut-off: {MIN_IMAGES} images", color=INK2, fontsize=8, va="center")
ax.set_ylim(-2.2, len(rows) - 0.4)
for i, (n, v) in enumerate(zip(names, vals)):
    if n in HAB:
        ax.text(v * 1.08, i, f"{v:,}", va="center", fontsize=8, color=INK)
ax.xaxis.set_major_formatter(thousands)
ax.set_xlabel("Images (log scale)")
ax.grid(axis="x", color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.tick_params(axis="y", labelsize=8, length=0)
ax.legend(handles=[Patch(color=C_HAB, label="HAB class"), Patch(color=C_NON, label="non-HAB class"),
                   Patch(color=C_DROP, label="dropped (< 100 images or beads)")],
          loc="lower right", frameon=False, fontsize=8)
ax.set_title("Class Distribution: SYKE 2016\u20132019", loc="left", fontsize=12, fontweight="bold")
save_show(fig, "class_dist_2016_2019.png")

### A3. Images per taxonomic group (2016–2019)

In [ ]:
groups = {}
for name, g, n_train, _ in rows:
    groups.setdefault(g, {"HAB": 0, "non-HAB": 0, "dropped": 0})[status(name, n_train)] += n_train
order = sorted(groups, key=lambda g: sum(groups[g].values()))
fig, ax = plt.subplots(figsize=(8, 4.2))
left = [0] * len(order)
for key, lab in [("HAB", "HAB classes"), ("non-HAB", "non-HAB classes"), ("dropped", "dropped classes")]:
    v = [groups[g][key] for g in order]
    ax.barh(order, v, left=left, color=COLOURS[key], label=lab, height=0.65, edgecolor=SURF, linewidth=1.5)
    left = [a + b for a, b in zip(left, v)]
for i, t in enumerate(left):
    ax.text(t + total_train * 0.006, i, f"{t:,}", va="center", fontsize=8, color=INK2)
ax.xaxis.set_major_formatter(thousands)
ax.set_xlabel("Images")
ax.grid(axis="x", color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.tick_params(axis="y", length=0)
ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.set_title("SYKE 2016\u20132019: images per taxonomic group", loc="left", fontsize=12, fontweight="bold")
save_show(fig, "taxonomic_groups_2016_2019.png")

### A4. Class imbalance (cumulative share, 2016–2019)

In [ ]:
cum, s = [], 0
for c in counts:
    s += c; cum.append(100 * s / total_train)
rank = list(range(1, len(counts) + 1))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(rank, cum, color=C_NON, lw=2)
ax.scatter(rank, cum, s=14, color=[colour(r[0], r[2]) for r in rows], zorder=3, edgecolor=SURF, linewidth=0.8)
ax.annotate(f"top 5 classes: {cum[4]:.0f}% of images", xy=(5, cum[4]), xytext=(12, cum[4] - 22),
            fontsize=9, color=INK, arrowprops=dict(arrowstyle="-", color=INK2, lw=0.8))
first_small = next(i for i, c in enumerate(counts) if c < MIN_IMAGES)
ax.axvline(first_small + 0.5, color=INK2, ls="--", lw=1)
ax.text(first_small + 1, 40, f"last {len(counts) - first_small} classes\n(< {MIN_IMAGES} images)\n"
        f"= {100 - cum[first_small - 1]:.1f}% of images", fontsize=8, color=INK2)
ax.set_xlabel("Classes ranked by size (largest first)")
ax.set_ylabel("Cumulative share of images (%)")
ax.set_ylim(0, 102); ax.set_xlim(0, len(counts) + 1)
ax.grid(color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.set_title("SYKE 2016\u20132019: class imbalance", loc="left", fontsize=12, fontweight="bold")
save_show(fig, "imbalance_2016_2019.png")

### A5. Class mix: training (2016–2019) vs test (2021)
Points far below the diagonal are classes that are much rarer in the 2021 test set than in training.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6.5))
FLOOR = 0.0008  # % shown for classes with 0 test images (1 image = 0.0017%)
lo, hi = FLOOR * 0.7, 40
ax.plot([lo, hi], [lo, hi], color=INK2, lw=1, ls="--")
ax.text(0.0016, 0.0042, "same share in both periods", fontsize=8, color=INK2, rotation=45, rotation_mode="anchor")
SHORT = {"Dolichospermum / Anabaenopsis": "Dolichospermum", "Snowella / Woronichinia": "Snowella/Woronichinia",
         "Aphanizomenon flosaquae": "Aphanizomenon", "Nodularia spumigena": "Nodularia",
         "Dinophysis acuminata": "Dinophysis", "Prorocentrum cordatum": "Prorocentrum"}
for name, g, n_train, n_test in rows:
    st = status(name, n_train)
    x = 100 * n_train / total_train
    y = max(100 * n_test / total_test, FLOOR)
    ax.scatter(x, y, s=46 if st == "HAB" else 26, color=colour(name, n_train), zorder=3,
               edgecolor=SURF, linewidth=1, marker="v" if n_test == 0 else "o")
    if st == "HAB":
        short = name.replace("Dolichospermum / Anabaenopsis coiled", "Dolichospermum coiled")
        short = SHORT.get(short, short)
        ax.annotate(f"{short} ({n_test:,})", (x, y), xytext=(6, -11) if short == "Nodularia" else (5, 3),
                    textcoords="offset points", fontsize=8, color=INK)
ax.axhline(FLOOR, color=GRID, lw=0.8)
ax.text(lo * 1.15, FLOOR, "0 test images", fontsize=7, color=INK2, va="center", backgroundcolor=SURF)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
pct = FuncFormatter(lambda v, _: f"{v:g}%")
ax.xaxis.set_major_formatter(pct); ax.yaxis.set_major_formatter(pct)
ax.set_xlabel("Share of 2016\u20132019 training images")
ax.set_ylabel("Share of 2021 classified test images")
ax.grid(color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.legend(handles=[Patch(color=C_HAB, label="HAB class (label: 2021 count)"),
                   Patch(color=C_NON, label="non-HAB class"), Patch(color=C_DROP, label="dropped class")],
          loc="upper left", frameon=False, fontsize=8)
ax.set_title("Class mix: training (2016\u20132019) vs test (2021)", loc="left", fontsize=11, fontweight="bold")
save_show(fig, "train_vs_test_mix.png")

## Part B. 2021 class distribution (Appendix Figure A2)
### B1. Count images per class folder
Counts `.png` **file names** only (no images are opened, so cloud-synced files are not downloaded).
Folders are counted in parallel threads, which speeds up Google Drive / OneDrive folders a lot.
If `DATA_2021` is not found, the Table 1 counts from the class table are used instead (they match the folder counts).

In [ ]:
HAB_KEYS = ["dolichospermum", "aphanizomenon", "oscillatoriales", "snowella",
            "nodularia", "dinophysis", "prorocentrum"]
DROP_KEYS = ["merismopedia", "chlorococcales", "licmophora", "gymnodiniales", "nitzschia",
             "katablepharis", "ceratoneis", "melosira", "aphanothece", "gonyaulax", "amylax", "beads"]

def kind(folder_name):
    n = folder_name.lower()
    if "unclass" in n:
        return "unclassifiable"
    if any(k in n for k in HAB_KEYS):
        return "HAB"
    if any(k in n for k in DROP_KEYS):
        return "dropped"
    return "non-HAB"

def count_png(folder):
    """Number of .png files directly inside one folder (file names only)."""
    with os.scandir(folder) as it:
        return sum(1 for e in it if e.is_file() and e.name.lower().endswith(".png"))

def class_folders(root):
    """Every folder under root that has no subfolders (= one class each)."""
    leaves = []
    for dirpath, subdirs, _ in os.walk(root):
        if not subdirs and Path(dirpath) != root:
            leaves.append(Path(dirpath))
    return leaves

t0 = time.time()
if DATA_2021.is_dir():
    folders = class_folders(DATA_2021)
    with ThreadPoolExecutor(max_workers=min(32, (os.cpu_count() or 4) * 2)) as pool:
        n_per = list(pool.map(count_png, folders))
    counts_2021 = {}
    for f, n in zip(folders, n_per):
        name = f.name if f.name not in counts_2021 else str(f.relative_to(DATA_2021))
        counts_2021[name] = n
    source = f"folder counts ({DATA_2021})"
else:
    counts_2021 = {r[0]: r[3] for r in DATA if r[3] > 0}   # 2021 has no folder for 0-image classes
    counts_2021["Unclassifiable"] = UNCLASSIFIABLE_2021
    source = "Kraft et al. (2022), Table 1 (folder not found)"

classes = sorted(counts_2021.items(), key=lambda kv: kv[1], reverse=True)
# class type: by keyword for folder names; by the class table for Table 1 names
table_names = {r[0]: r[2] for r in DATA}
def type_2021(name):
    if name in table_names:
        return status(name, table_names[name])
    return kind(name)

with open(OUT / "class_counts_2021.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["class", "images", "type"])
    for name, n in classes:
        w.writerow([name, n, type_2021(name)])

unc = sum(n for name, n in classes if type_2021(name) == "unclassifiable")
cls = [(name, n) for name, n in classes if type_2021(name) != "unclassifiable"]
total_cls = sum(n for _, n in cls)
hab_n = sum(n for name, n in cls if type_2021(name) == "HAB")
print(f"Source: {source}  ({time.time() - t0:.1f} s)")
print(f"{len(cls)} classes, {total_cls:,} classified images, {unc:,} unclassifiable "
      f"({100 * unc / max(total_cls + unc, 1):.0f}% of all images)")
print(f"HAB classes: {hab_n:,} images ({100 * hab_n / max(total_cls, 1):.1f}% of classified)")
print("Classes with < 10 images:", ", ".join(f"{n_} ({c})" for n_, c in cls if c < 10) or "none")

### B2. Class Distribution: SYKE 2021

In [ ]:
names = [n for n, _ in classes][::-1]
raw = [c for _, c in classes][::-1]
vals = [max(c, 0.8) for c in raw]            # 0 shown as a stub on the log axis
kinds = [type_2021(n) for n in names]
fig, ax = plt.subplots(figsize=(8, max(6, 0.22 * len(classes) + 1.5)))
bars = ax.barh(names, vals, color=[COLOURS[k] for k in kinds], height=0.72)
for b, k in zip(bars, kinds):
    if k == "unclassifiable":
        b.set_hatch("///"); b.set_edgecolor("white")
for i, (v, r, k) in enumerate(zip(vals, raw, kinds)):
    if k in ("HAB", "unclassifiable") or r < 10:
        ax.text(v * 1.08, i, f"{r:,}", va="center", fontsize=7.5, color=INK)
ax.set_xscale("log"); ax.set_xlim(left=0.7)
ax.xaxis.set_major_formatter(thousands)
ax.set_xlabel("Images (log scale)")
ax.grid(axis="x", color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.tick_params(axis="y", labelsize=7.5, length=0)
ax.legend(handles=[Patch(color=C_HAB, label="HAB class"), Patch(color=C_NON, label="non-HAB class"),
                   Patch(color=C_DROP, label="dropped class (< 100 images or beads)"),
                   Patch(facecolor=C_UNC, hatch="///", edgecolor="white", label="unclassifiable")],
          loc="lower right", frameon=False, fontsize=8)
ax.set_title("Class Distribution: SYKE 2021", loc="left", fontsize=11, fontweight="bold")
save_show(fig, "class_dist_2021.png")
print("Saved to", OUT.resolve())